# 69 Optode co-registration visual check

Visual sanity check for the original `_validator_coreg.py` validator
(removed in commit `816a7bb`, output preserved in
`thesis_results_out/coreg_invariance.csv`). For one subject, run
`ColoredStickerProcessor` with the full three-colour HSV configuration
the validator used (yellow optodes 'O', green detectors 'G', magenta
sources 'M'), match per group with the validator's greedy strict-1:1
pairing capped at 10 mm, and render everything in two linked PyVista
panels.

Detection ranges, optode length, matching algorithm, and 10 mm
face-spurious cap mirror `_validator_coreg.py` exactly. The per-group
deviation stats printed in cell 5 should reproduce the corresponding
rows of `coreg_invariance.csv`. If they do, the side-by-side plot tells
you what those numbers mean geometrically.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pyvista as pv

import cedalion
import cedalion.vis.blocks as vbx
from cedalion.geometry.landmarks import normalize_landmarks_labels
from cedalion.geometry.photogrammetry.processors import ColoredStickerProcessor

from _thesis_data import load_raw, load_anon, load_landmarks

pv.set_jupyter_backend('server')

SUBJECT = 17

# Exact HSV ranges from _validator_coreg.py (commit 68ee3d5):
#   O = yellow optode stickers (legacy notebook 41 default)
#   G = green detector stickers
#   M = magenta source stickers
COLORS = {
    'O': (0.11, 0.21, 0.70, 1.0),
    'G': (0.14, 0.30, 0.30, 1.0),
    'M': (0.85, 1.00, 0.40, 1.0),
}
OPTODE_LENGTH = 22.6 * cedalion.units.mm
L = float(OPTODE_LENGTH.to('mm').magnitude)

# Validator drops orig-side detections without an anon counterpart
# within this radius (face-region false positives).
FACE_SPURIOUS_RADIUS_MM = 10.0

# Visual identity per group: red-ish for O, green for G, magenta for M.
GROUP_COLOR = {'O': '#ffaa00', 'G': '#22cc22', 'M': '#cc22cc'}


## 1. Load surfaces and landmarks

Subject17 is the post-dev-merge verified subject. Change `SUBJECT` in
the previous cell to re-run on any other optode-cohort subject
(16, 18, 19, 20, 21, 22).


In [2]:
surface_orig = load_raw(SUBJECT)
surface_anon = load_anon(SUBJECT)
landmarks = normalize_landmarks_labels(load_landmarks(SUBJECT))

print(f'Subject{SUBJECT}')
print(f'  original  vertices: {len(surface_orig.mesh.vertices):,}')
print(f'  anonymized vertices: {len(surface_anon.mesh.vertices):,}')
print(f'  landmarks: {list(landmarks.label.values)}')


Subject17
  original  vertices: 1,284,667
  anonymized vertices: 593,963
  landmarks: [np.str_('Nz'), np.str_('Iz'), np.str_('Cz'), np.str_('LPA'), np.str_('RPA')]


## 2. Detect stickers on both scans, split by colour group

One `ColoredStickerProcessor.process` call per surface with all three
HSV ranges, then split the returned centres / normals by their
`group` coordinate. Same call shape `_validator_coreg.py` used.


In [3]:
def detect_by_group(surface):
    proc = ColoredStickerProcessor(colors=COLORS)
    centres, normals = proc.process(surface, details=False)
    if len(centres) == 0:
        return {}
    c_np = centres.pint.dequantify().values
    n_np = normals.values
    g_arr = np.asarray(centres.group.values)
    out = {}
    for g in np.unique(g_arr):
        m = g_arr == g
        out[str(g)] = {'centres': c_np[m], 'normals': n_np[m]}
    return out

groups_orig = detect_by_group(surface_orig)
groups_anon = detect_by_group(surface_anon)

for g in sorted(set(groups_orig) | set(groups_anon)):
    no = len(groups_orig.get(g, {'centres': []})['centres'])
    na = len(groups_anon.get(g, {'centres': []})['centres'])
    print(f'  group {g}: original={no:3d}, anonymized={na:3d}')


[[-62.126129 379.344299 496.446136]
 [-60.2271   380.594482 495.111847]
 [-60.2271   380.594482 495.111847]
 ...
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]
 [ 63.561325  90.3022   582.236816]]
[[173.067749 152.878815 524.397705]
 [171.551178 156.72226  523.730469]
 [195.392883 145.783371 477.122894]
 ...
 [168.050629   2.322266 444.530457]
 [168.193817   2.722275 444.530457]
 [192.774017  53.522278 444.530457]]
O (0.11, 0.21, 0.7, 1.0)
[[-62.126129 379.344299 496.446136]
 [-60.2271   380.594482 495.111847]
 [-60.2271   380.594482 495.111847]
 ...
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]
 [ 63.561325  90.3022   582.236816]]
[[172.471527 177.925201 494.870483]
 [186.736481  67.19873  514.295105]
 [140.480835 -19.345001 457.866455]
 ...
 [187.237213  33.922272 444.530457]
 [187.257904  33.522263 444.530457]
 [187.015991  33.060547 444.626129]]
G (0.14, 0.3, 0.3, 1.0)


[[-62.126129 379.344299 496.446136]
 [-60.2271   380.594482 495.111847]
 [-60.2271   380.594482 495.111847]
 ...
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]
 [ 63.561325  90.3022   582.236816]]
[[ 104.91983   218.959656  476.566742]
 [-239.395691  -25.133606  518.167358]
 [-201.389893  -14.077728  528.530457]
 ...
 [-185.138641  -18.929413  583.291687]
 [-244.212967  -44.16188   582.381287]
 [-240.31311   -41.677734  582.530457]]
M (0.85, 1.0, 0.4, 1.0)
[0.02940559 0.95909248 2.11015511 0.7870147  0.90299782 1.0188369
 0.78557518 0.22357185 0.38533485 0.09604622 1.72756279 0.34873634
 1.88083138 0.87307441 0.52633369 0.63046943 1.3329522  1.06282769
 0.47782952 2.15998284 0.06028175 0.34719841 0.55752113 0.26658088
 0.20127052 0.08177361 0.48711632 0.78656883 0.00957512 0.89898262
 0.37162316 1.8688401  0.91721184 0.35123298 0.19180962 0.15842186
 0.15517921 0.07921333 0.24855446 0.37246666 0.95909691 0.55750252
 1.88079354 0.43935707 0.21303124 0.22787727 1

/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


[[181.585419  72.619293 523.590393]
 [170.857559 157.88739  523.812012]
 [181.301056  72.322266 523.730469]
 ...
 [-45.646057 139.922272 582.130432]
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]]
[[173.067749 152.878815 524.397705]
 [171.551178 156.72226  523.730469]
 [173.665268 152.718643 523.777771]
 ...
 [191.205231  50.322266 444.530457]
 [191.330841  50.72226  444.530457]
 [192.774017  53.522278 444.530457]]
O (0.11, 0.21, 0.7, 1.0)
[[181.585419  72.619293 523.590393]
 [170.857559 157.88739  523.812012]
 [181.301056  72.322266 523.730469]
 ...
 [-45.646057 139.922272 582.130432]
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]]
[[172.471527 177.925201 494.870483]
 [186.736481  67.19873  514.295105]
 [170.163727 158.72226  523.730469]
 ...
 [187.237213  33.922272 444.530457]
 [187.257904  33.522263 444.530457]
 [187.015991  33.060547 444.626129]]
G (0.14, 0.3, 0.3, 1.0)


[[181.585419  72.619293 523.590393]
 [170.857559 157.88739  523.812012]
 [181.301056  72.322266 523.730469]
 ...
 [-45.646057 139.922272 582.130432]
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]]
[[104.91983  218.959656 476.566742]
 [117.292023 200.184784 514.930481]
 [116.492035 200.297455 514.930481]
 ...
 [187.158783  33.927765 459.793518]
 [130.975403 209.783035 430.359772]
 [191.995453 154.842987 446.893677]]
M (0.85, 1.0, 0.4, 1.0)
[0.32077221 1.00696094 2.0571394  0.88362262 1.35326367 1.04817852
 0.23729847 0.33489453 0.20170239 1.14434033 1.23635235 0.41266803
 1.97634503 0.85782795 0.5323161  0.81556507 1.38189709 1.25476056
 0.35434613 2.08681897 0.09497632 0.21938305 0.75328479 0.24899768
 0.40391986 0.12973156 0.22483471 1.82561673 0.5472395  0.14191906
 0.77463825 0.28751934 1.87140308 0.81853802 0.11845935 0.13803662
 0.02745197 0.21372058 0.44797046 0.09909222 0.92190472 0.60756336
 0.14160053 1.89238125 0.6579198  0.25093738 1.74967762 0.10776

/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


## 3. Per-group greedy strict-1:1 matching (10 mm cap)

Mirrors `match_one_to_one` from `_validator_coreg.py`: sort all
pairwise distances ascending, greedily pick the smallest-distance
pair whose endpoints are still free, stop when no remaining pair is
within `FACE_SPURIOUS_RADIUS_MM`. Strict 1:1 by construction; pairs
beyond 10 mm are treated as face-region false positives and dropped.
Per-group `sticker_dev_mm` and `scalp_dev_mm` should reproduce the
validator's CSV rows for this subject.


In [4]:
def match_one_to_one(c_orig, c_anon, max_distance_mm=FACE_SPURIOUS_RADIUS_MM):
    if len(c_orig) == 0 or len(c_anon) == 0:
        return np.empty(0, dtype=int), np.empty(0, dtype=int)
    cost = np.linalg.norm(c_orig[:, None, :] - c_anon[None, :, :], axis=-1)
    n_a = cost.shape[1]
    order = np.argsort(cost, axis=None)
    used_o, used_a = set(), set()
    pairs = []
    for k in order:
        if cost.flat[k] > max_distance_mm:
            break
        i, j = divmod(int(k), n_a)
        if i in used_o or j in used_a:
            continue
        pairs.append((i, j))
        used_o.add(i)
        used_a.add(j)
    if not pairs:
        return np.empty(0, dtype=int), np.empty(0, dtype=int)
    i_o, i_a = map(np.array, zip(*pairs))
    return i_o, i_a

matches = {}  # group -> dict of arrays
print(f'{"group":>5}  {"orig":>4}  {"anon":>4}  {"paired":>6}'
      f'  {"st_mean":>8}  {"st_max":>7}  {"sc_mean":>8}  {"sc_max":>7}')
for g in sorted(set(groups_orig) & set(groups_anon)):
    co, no = groups_orig[g]['centres'], groups_orig[g]['normals']
    ca, na = groups_anon[g]['centres'], groups_anon[g]['normals']
    i_o, i_a = match_one_to_one(co, ca)
    if len(i_o) == 0:
        print(f'{g:>5}  {len(co):>4d}  {len(ca):>4d}  {0:>6d}'
              f'  {"-":>8}  {"-":>7}  {"-":>8}  {"-":>7}')
        continue
    co_p, no_p = co[i_o], no[i_o]
    ca_p, na_p = ca[i_a], na[i_a]
    sticker_dev = np.linalg.norm(co_p - ca_p, axis=1)
    scalp_dev = np.linalg.norm((co_p - L * no_p) - (ca_p - L * na_p), axis=1)
    matches[g] = {
        'i_o': i_o, 'i_a': i_a,
        'co_p': co_p, 'ca_p': ca_p,
        'scalp_o': co_p - L * no_p, 'scalp_a': ca_p - L * na_p,
        'sticker_dev': sticker_dev, 'scalp_dev': scalp_dev,
    }
    print(f'{g:>5}  {len(co):>4d}  {len(ca):>4d}  {len(i_o):>6d}'
          f'  {sticker_dev.mean():>8.4f}  {sticker_dev.max():>7.4f}'
          f'  {scalp_dev.mean():>8.4f}  {scalp_dev.max():>7.4f}')


group  orig  anon  paired   st_mean   st_max   sc_mean   sc_max
    G   126   126     125    0.3178   2.3394    0.3162   1.2817
    M     2     2       2    0.2917   0.4953    0.2646   0.4787
    O    21    21      21    0.3845   1.5088    0.3654   1.3907


## 4. Side-by-side detections (all three colour groups)

Two linked PyVista panels. Left: original. Right: anonymized.

- **Yellow / green / magenta spheres** -- raw
  `ColoredStickerProcessor` detections, coloured by their group
  (`O` / `G` / `M`).
- **White spheres** -- scalp-projected positions (centre - 22.6 mm
  along the surface normal); these are what the validator's
  `scalp_dev` actually compares.
- **Cyan labelled spheres** -- five 10-20 landmarks for orientation.
- **Arrows** -- surface normals at each detected sticker.

`link_views()` ties the cameras so rotating one panel rotates the
other.


In [5]:
pvplt = pv.Plotter(shape=(1, 2), window_size=(1600, 800))

for col, (label, surface, groups) in enumerate([
    ('Original', surface_orig, groups_orig),
    ('Anonymized', surface_anon, groups_anon),
]):
    pvplt.subplot(0, col)
    counts = ', '.join(f'{g}={len(groups.get(g, {"centres": []})["centres"])}'
                       for g in ['O', 'G', 'M'])
    pvplt.add_text(f'{label} (Subject{SUBJECT}) -- {counts}', font_size=10)
    vbx.plot_surface(pvplt, surface, opacity=1.0)
    for g, data in groups.items():
        c, n = data['centres'], data['normals']
        if len(c) == 0:
            continue
        pvplt.add_points(c, color=GROUP_COLOR.get(g, 'red'),
                         point_size=14, render_points_as_spheres=True)
        scalp = c - L * n
        pvplt.add_points(scalp, color='white',
                         point_size=8, render_points_as_spheres=True)
    vbx.plot_labeled_points(pvplt, landmarks, color='cyan', show_labels=True)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:44959/index.html?ui=P_0x7f8d53276e10_0&reconnect=auto" class="pyvi…

## 5. Per-group matching diagnostic

For each matched pair within a group, draw a white line segment from
the original sticker centre (group-coloured) to its paired anon
centre (cyan). Healthy match: coincident dots, invisible lines.
Degenerate match: long visible white lines, or unpaired group-coloured
dots with no cyan counterpart (those are face-region false positives
the validator dropped via the 10 mm cap).


In [6]:
pvplt = pv.Plotter(shape=(1, 2), window_size=(1600, 800))

for col, (label, surface) in enumerate([
    ('Original surface + matches', surface_orig),
    ('Anonymized surface + matches', surface_anon),
]):
    pvplt.subplot(0, col)
    pvplt.add_text(f'{label} (Subject{SUBJECT})', font_size=10)
    vbx.plot_surface(pvplt, surface, opacity=0.5)
    for g, m in matches.items():
        pvplt.add_points(m['co_p'], color=GROUP_COLOR.get(g, 'red'),
                         point_size=12, render_points_as_spheres=True)
        pvplt.add_points(m['ca_p'], color='cyan',
                         point_size=10, render_points_as_spheres=True)
        for a, b in zip(m['co_p'], m['ca_p']):
            if np.linalg.norm(a - b) > 1e-6:
                pvplt.add_mesh(pv.Line(a, b), color='white', line_width=2)
    # Unpaired originals (face-region false positives dropped by 10 mm cap)
    for g in groups_orig:
        co = groups_orig[g]['centres']
        paired = matches.get(g, {}).get('i_o', np.empty(0, dtype=int))
        unpaired_mask = np.ones(len(co), dtype=bool)
        if len(paired):
            unpaired_mask[paired] = False
        if unpaired_mask.any():
            pvplt.add_points(co[unpaired_mask], color='black',
                             point_size=14, render_points_as_spheres=True)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:44959/index.html?ui=P_0x7f8d55df52d0_1&reconnect=auto" class="pyvi…

## 6. What to look for

**Validator passes if all of:**

- Per-group sticker counts in section 2 differ by at most a couple
  between original and anonymized (the cap is outside the deletion
  mask, so detection should be near-identical).
- Per-group `st_mean` and `sc_mean` in section 3 are sub-millimetre
  (matches the corresponding rows of
  `thesis_results_out/coreg_invariance.csv`).
- Section 4 panels show the same sticker pattern in both, no
  coloured spheres on the deleted face area in the anonymized panel.
- Section 5 shows no long white lines, and no black spheres on the
  cap (a black sphere on the cap means a real cap sticker lost its
  partner -- problem; a black sphere in the deleted face region is
  expected and harmless, those are exactly the false positives the
  10 mm cap is designed to drop).

**Validator is suspect if any of:**

- Per-group counts differ by more than a couple.
- Group means exceed a few mm despite the cap being outside the mask.
- A coloured sphere appears on the anonymized scan in a region where
  the original mesh has no sticker.
- A black sphere sits on the cap (real cap detection lost its partner).
